# 1D Diffusion

In the previous two lessons, we studied the numerical solution of the linear and nonlinear convection equations and learned about the CFL stability condition. We now turn to the **one-dimensional diffusion equation**:

$$
\label{eq-1d-diffusion-pde}
\frac{\partial u}{\partial t}=\nu\frac{\partial^2 u}{\partial x^2}.
$$

Here, $\nu>0$ is a constant *diffusion coefficient*, with units of length squared over time. This equation describes how spatial variations in $u$ smooth out over time. It applies to several physical settings, like:

- the evolution of temperature along a rod, where it is called the *heat equation*;
- the diffusion of a pollutant along a pipe containing still fluid; or
- the diffusion of a dissolved substance through a stationary gel.

On a finite domain, its solution needs an initial profile and a boundary condition at each end. In this lesson, we use $0\le x\le L$, with $L=2$, and hold both endpoint values fixed:

$$
\label{eq-1d-diffusion-boundaries}
u(0,t)=u(L,t)=1,\qquad t\ge0.
$$

These prescribed values are called *Dirichlet boundary conditions*. We will specify the initial profile below.

This is the first instance where we encounter a **second-order spatial derivative**. Our first-derivative schemes do not directly provide an approximation for this term, so we need to derive a new finite-difference formula before advancing the solution in time.

## Discretize the second derivative

The second spatial derivative measures how the *slope* of $u$ changes with position. To approximate it, we use the value at a grid point and its two immediate neighbors.

As in [Lesson 6](./06-1d-convection.ipynb), take a uniform grid $x_i=i\Delta x$, with $i=0,\ldots,N$ and $\Delta x=L/N$. At a fixed time, write $u_i=u(x_i,t)$ for the exact values while deriving the spatial approximation. For a sufficiently smooth function, Taylor expansion about $x_i$ gives

$$
\label{eq-diffusion-taylor-pair}
\begin{aligned}
u_{i+1} &= u_i+\Delta x\left.\frac{\partial u}{\partial x}\right|_i
+\frac{\Delta x^2}{2}\left.\frac{\partial^2 u}{\partial x^2}\right|_i
+\frac{\Delta x^3}{6}\left.\frac{\partial^3 u}{\partial x^3}\right|_i
+\mathcal{O}(\Delta x^4),\\
u_{i-1} &= u_i-\Delta x\left.\frac{\partial u}{\partial x}\right|_i
+\frac{\Delta x^2}{2}\left.\frac{\partial^2 u}{\partial x^2}\right|_i
-\frac{\Delta x^3}{6}\left.\frac{\partial^3 u}{\partial x^3}\right|_i
+\mathcal{O}(\Delta x^4).
\end{aligned}
$$

Adding the expansions cancels the displayed odd-derivative terms and gives:

$$
u_{i+1}+u_{i-1}=2u_i+\Delta x^2\left.\frac{\partial^2 u}{\partial x^2}\right|_i
+\mathcal{O}(\Delta x^4).
$$

Subtract $2u_i$ and divide by $\Delta x^2$ to obtain

$$
\label{eq-diffusion-centered-second-derivative}
\left.\frac{\partial^2 u}{\partial x^2}\right|_i
=\frac{u_{i+1}-2u_i+u_{i-1}}{\Delta x^2}
+\mathcal{O}(\Delta x^2).
$$

This is a **second-order accurate central difference**: for a smooth profile, its leading error scales with $\Delta x^2$. Notice that dividing by $\Delta x^2$ also changes the order of the remainder from $\mathcal{O}(\Delta x^4)$ to $\mathcal{O}(\Delta x^2)$.

:::{warning .simple .dropdown icon=false open=false} On paper
Reconstruct the derivation by adding the two Taylor expansions. Then consider the numerator $u_{i+1}-2u_i+u_{i-1}$: what sign does it have at a strict local maximum? What value does it have when the three points lie on a straight line? Use the diffusion equation to predict the corresponding change in $u_i$.
:::

## Forward Euler in time

Let $u_i^n$ denote our numerical approximation to $u(x_i,t^n)$, where $t^n=n\Delta t$. Using the central difference in [Equation %s](#eq-diffusion-centered-second-derivative) for the spatial derivative, we apply **forward Euler** to advance in time. As in our earlier ODE calculations, Euler's method evaluates the rate of change using the current state:

$$
\label{eq-diffusion-ftcs}
\frac{u_i^{n+1}-u_i^n}{\Delta t}
=\nu\frac{u_{i+1}^n-2u_i^n+u_{i-1}^n}{\Delta x^2}.
$$

Rearranging gives the explicit update

$$
\label{eq-diffusion-explicit-update}
u_i^{n+1}=u_i^n+\frac{\nu\Delta t}{\Delta x^2}
\left(u_{i+1}^n-2u_i^n+u_{i-1}^n\right),
\qquad i=1,\ldots,N-1.
$$

Every value on the right-hand side belongs to the old time level $n$. The new interior values can therefore be calculated directly from the old array. This combination is called **forward-time, central-space (FTCS)**. It is first-order accurate in time and second-order accurate in space; these orders describe the truncation error for sufficiently smooth solutions.

The three-point formula applies to the interior points. At the two endpoints, we impose the prescribed boundary values instead:

$$
u_0^{n+1}=u_N^{n+1}=1.
$$

For the initial profile, we reuse the square pulse from the convection lesson:

$$
\label{eq-diffusion-initial-profile}
u(x,0)=
\begin{cases}
2, & 0.5\le x\le1,\\
1, & \text{elsewhere on }[0,2].
\end{cases}
$$

This profile makes the spreading of the pulse easy to see. Its jumps do not satisfy the smoothness assumption used in the Taylor derivation, however, so we should not use them to infer the formal order of accuracy.

:::{warning .simple .dropdown icon=false open=false} On paper
For the three values $[1,2,1]$ and $\nu\Delta t/\Delta x^2=0.2$, calculate the updated middle value while holding the endpoints fixed. Does the change agree with your prediction at a local maximum?
:::

## Stability through convex weights

In [Lesson 7](./07-cfl-condition.ipynb), we established a stability bound by writing the convection update as a weighted average. We can reuse that argument for diffusion, now with three old values instead of two.

### Read the weights

Define the **diffusion number**

$$
\label{eq-diffusion-number}
r=\frac{\nu\Delta t}{\Delta x^2}.
$$

This ratio is dimensionless: $\nu$ has units of length squared per time. Collecting terms in [Equation %s](#eq-diffusion-explicit-update) gives

$$
\label{eq-diffusion-convex-update}
u_i^{n+1}=r u_{i-1}^n+(1-2r)u_i^n+r u_{i+1}^n.
$$

The weights sum to one. They are all nonnegative when

$$
\label{eq-diffusion-convex-condition}
0\le r\le\frac12.
$$

Under this condition, the new value is a *convex combination*: it lies between the smallest and largest of the three old values. With our fixed endpoint values, the update cannot create a value outside the range of the initial and boundary data. For the square pulse, that range is $[1,2]$.

This explains a useful property of the solution, but stability asks about the amplification of a disturbance. As in Lesson 7, we apply the argument to the difference between two calculations.

### Bound a disturbance

Consider two numerical solutions with the same grid, diffusivity, time step, and prescribed endpoint values, but slightly different initial profiles. Define

$$
\label{eq-diffusion-perturbation}
e_i^n=u_{b,i}^n-u_{a,i}^n,
\qquad E^n=\max_i|e_i^n|.
$$

Here, $e_i^n$ is the difference between two numerical solutions, not the error relative to an exact solution. Subtracting their updates gives

$$
\label{eq-diffusion-perturbation-update}
e_i^{n+1}=r e_{i-1}^n+(1-2r)e_i^n+r e_{i+1}^n.
$$

For $0\le r\le1/2$, the triangle inequality and the nonnegative weights imply

$$
\label{eq-diffusion-perturbation-bound}
\begin{aligned}
|e_i^{n+1}|
&\le r|e_{i-1}^n|+(1-2r)|e_i^n|+r|e_{i+1}^n|\\
&\le rE^n+(1-2r)E^n+rE^n\\
&=E^n.
\end{aligned}
$$

Both endpoint differences are zero because the prescribed boundary values are identical. Taking the maximum over the whole array and repeating the bound over successive steps therefore yields

$$
\label{eq-diffusion-stability-bound}
E^n\le E^{n-1}\le\cdots\le E^0.
$$

The amplification bound is one, independent of the grid spacing and number of steps. This establishes stability for the stated update and boundary treatment when $0\le r\le1/2$. It does not establish that the numerical solution is sufficiently accurate.

### What happens beyond the limit?

When $r>1/2$, the middle weight is negative. The argument above no longer applies, but a failed proof is not itself proof of growth. To see a mechanism for growth, reuse the alternating disturbance from Lesson 7. On a grid without boundaries, let

$$
e_i^n=A_n(-1)^i.
$$

Both neighbors of the point $i$ have the opposite sign. Substitution into [Equation %s](#eq-diffusion-perturbation-update) gives

$$
\label{eq-diffusion-alternating-factor}
A_{n+1}=(1-4r)A_n.
$$

For $r>1/2$, the factor $1-4r$ is less than $-1$: the disturbance reverses sign and grows in magnitude at every step. For example, $r=0.6$ gives a factor of $-1.4$. At $r=1/2$, the factor is $-1$, so this particular pattern changes sign without decaying. Stability permits that behavior; it does not require every disturbance to decay.

Our two solutions have identical fixed endpoint values, so their difference is zero at both ends. The alternating disturbance therefore stops at the boundaries. Each update uses only neighboring values, so the boundary’s effect moves inward by at most one grid point per step. For example, over five steps, the factor $1-4r$ predicts the disturbance exactly at points more than five grid intervals from either end. Closer to the endpoints, the disturbance no longer follows this simple alternating pattern.

### The time step now scales with the square of the spacing

For positive diffusivity, the bound gives

$$
\label{eq-diffusion-time-step-limit}
\Delta t\le\frac{\Delta x^2}{2\nu}.
$$

Halving $\Delta x$ therefore quarters the largest permitted time step. Keeping the same diffusion number on the finer grid requires four times as many steps to reach the same physical time. In contrast, the convection time-step limit in Lesson 7 scaled with $\Delta x$. The diffusion number measures a different balance from the distance-per-step interpretation of the Courant number.

:::{warning .simple .dropdown icon=false open=false} On paper
1. Reconstruct the bound on $E^{n+1}$, identifying where nonnegative weights and identical boundary data enter the argument.
2. Calculate the three weights and the alternating-disturbance factor for $r=0.2$, $0.5$, and $0.6$. Which cases control the magnitude of that disturbance?
3. If you halve $\Delta x$ while keeping $\Delta t$ fixed, what happens to $r$? Starting from $r=0.2$, does the refined calculation still satisfy the bound?
:::

### And solve!

 We are ready to number-crunch!

The next two code cells initialize the problem by loading the needed libraries, then defining the solution parameters and initial condition. This time, we don't let the user choose just *any* $\Delta t$, though; we have decided this is not safe: people just like to blow things up. Instead, the code calculates a value of $\Delta t$ that will be in the stable range, according to the spatial discretization chosen! You can now experiment with different solution parameters to see how the numerical solution changes, but it won't blow up.

In [ ]:
import numpy
from matplotlib import pyplot
%matplotlib inline

In [ ]:
# Set the font family and size to use for Matplotlib figures.
pyplot.rcParams['font.family'] = 'serif'
pyplot.rcParams['font.size'] = 16

In [ ]:
# Set parameters.
nx = 41  # number spatial grid points
L = 2.0  # length of the domain
dx = L / (nx - 1)  # spatial grid size
nu = 0.3  # viscosity
sigma = 0.2  # CFL limit
dt = sigma * dx**2 / nu  # time-step size
nt = 20  # number of time steps to compute

# Get the grid point coordinates.
x = numpy.linspace(0.0, L, num=nx)

# Set the initial conditions.
u0 = numpy.ones(nx)
mask = numpy.where(numpy.logical_and(x >= 0.5, x <= 1.0))
u0[mask] = 2.0

In [ ]:
# Integrate in time.
u = u0.copy()
for n in range(nt):
    u[1:-1] = u[1:-1] + nu * dt / dx**2 * (u[2:] - 2 * u[1:-1] + u[:-2])

In [ ]:
# Plot the solution after nt time steps
# along with the initial conditions.
pyplot.figure(figsize=(6.0, 4.0))
pyplot.xlabel('x')
pyplot.ylabel('u')
pyplot.grid()
pyplot.plot(x, u0, label='Initial',
            color='C0', linestyle='--', linewidth=2)
pyplot.plot(x, u, label='nt = {}'.format(nt),
            color='C1', linestyle='-', linewidth=2)
pyplot.legend(loc='upper right')
pyplot.xlim(0.0, L)
pyplot.ylim(0.5, 2.5);

## Animations

Looking at before-and-after plots of the wave in motion is helpful, but it's even better if we can see it changing! 

First, let's import the `animation` module of `matplotlib` as well as a special IPython display method called `HTML` (more on this in a bit).

##### Note

You will also have to install a video encoder/decoder named `ffmpeg`.

If you use Linux or OSX, you can install ffmpeg using conda:
```
conda install -c conda-forge ffmpeg
```

If you use Windows, installation instructions can be found [here](http://adaptivesamples.com/how-to-install-ffmpeg-on-windows/).

In [ ]:
from matplotlib import animation
from IPython.display import HTML

We are going to create an animation.
This takes a few steps, but it's actually not hard to do!

First, we define a function, called `diffusion`, that computes the numerical solution of the 1D diffusion equation over the time steps.
(The function returns a list with `nt` elements, each one being a Numpy array.)

In [ ]:
def diffusion(u0, sigma=0.5, nt=20):
    """
    Computes the numerical solution of the 1D diffusion equation
    over the time steps.
    
    Parameters
    ----------
    u0 : numpy.ndarray
        The initial conditions as a 1D array of floats.
    sigma : float, optional
        The value of nu * dt / dx^2;
        default: 0.5.
    nt : integer, optional
        The number of time steps to compute;
        default: 20.
    
    Returns
    -------
    u_hist : list of numpy.ndarray objects
        The history of the numerical solution.
    """
    u_hist = [u0.copy()]
    u = u0.copy()
    for n in range(nt):
        u[1:-1] = u[1:-1] + sigma * (u[2:] - 2 * u[1:-1] + u[:-2])
        u_hist.append(u.copy())
    return u_hist

We now call the function to store the history of the solution:

In [ ]:
# Compute the history of the numerical solution.
u_hist = diffusion(u0, sigma=sigma, nt=nt)

Next, we create a Matplotlib figure that we want to animate.
For now, the figure contains the initial solution (our top-hat function).

In [ ]:
fig = pyplot.figure(figsize=(6.0, 4.0))
pyplot.xlabel('x')
pyplot.ylabel('u')
pyplot.grid()
line = pyplot.plot(x, u0,
                   color='C0', linestyle='-', linewidth=2)[0]
pyplot.xlim(0.0, L)
pyplot.ylim(0.5, 2.5)
fig.tight_layout()

**Note**: `pyplot.plot()` can (optionally) return several values. Since we're only creating one line, we ask it for the "zeroth" (and only...) line by adding `[0]` after the `pyplot.plot()` call.

Now that our figure is initialized, we define a function `update_plot` to update the data of the line plot based on the time-step index.

In [ ]:
def update_plot(n, u_hist):
    """
    Update the line y-data of the Matplotlib figure.
    
    Parameters
    ----------
    n : integer
        The time-step index.
    u_hist : list of numpy.ndarray objects
        The history of the numerical solution.
    """
    fig.suptitle('Time step {:0>2}'.format(n))
    line.set_ydata(u_hist[n])

Next, we create an `animation.FuncAnimation` object with the following arguments:

* `fig`: the name of our figure,
* `diffusion`: the name of our solver function,
* `frames`: the number of frames to dra (which we set equal to `nt`),
* `fargs`: extra arguments to pass to the function `diffusion`,
* `interval`: the number of milliseconds each frame appears for.

In [ ]:
# Create an animation.
anim = animation.FuncAnimation(fig, update_plot,
                               frames=nt, fargs=(u_hist,),
                               interval=100)

Ok! Time to display the animation.
We use the `HTML` display method that we imported above and the `to_html5_video` method of the animation object to make it web compatible.

In [ ]:
# Display the video.
HTML(anim.to_html5_video())